# 7.8 · 模型解释 / Model Interpretability (SHAP / LIME)

> **课程定位 / Where this fits**
> 第 8 课，**Part 7 · 模型评估与优化**。
> Lesson 8, **Part 7 · Model Evaluation & Tuning**.
>
> 随机森林、GBDT、神经网络都是"黑箱"——预测很准，但**说不清为什么**。在风控、医疗、招聘等场景，"为什么拒绝这个人"是法律和信任的硬要求。**可解释性**工具打开黑箱：**SHAP、LIME、置换重要性、PDP/ICE**。这是近年工业界和面试**越来越重视**的方向。
> Random forests, GBDT, neural nets are "black boxes" — accurate but **can't say why**. In risk, medicine, hiring, "why was this person rejected" is a legal and trust requirement. **Interpretability** tools open the box: **SHAP, LIME, permutation importance, PDP/ICE**. An increasingly valued area in industry and interviews.
>
> 💼 **实战/面试视角**："怎么解释黑箱模型 / SHAP 是什么 / 全局 vs 局部解释" 越来越高频。
> 💼 **Practical/interview angle:** "explaining a black box / what is SHAP / global vs local explanations" — increasingly common.

> 💡 **面试相关 / Interview-relevant**
> - "全局解释 vs 局部解释的区别"（出镜率 ★★★★）
> - "SHAP 的核心思想（Shapley 值/博弈论）"（★★★★★）
> - "SHAP vs LIME 区别"（★★★★）
> - "PDP/ICE 看什么"（★★★）
> - "置换重要性 vs 不纯度重要性"（★★★★，接 5.7）

---

## 学习目标 / Learning Objectives

1. 区分**全局**解释（整个模型）vs **局部**解释（单条预测）。
   Distinguish global (whole model) vs local (single prediction) explanations.
2. 用**置换重要性**做无偏的全局重要性。
   Use permutation importance for unbiased global importance.
3. 用 **PDP/ICE** 看特征如何影响预测。
   Use PDP/ICE to see how a feature affects predictions.
4. 用 **SHAP**（Shapley 值）做全局+局部统一解释。
   Use SHAP (Shapley values) for unified global+local explanations.
5. 用 **LIME** 做单条预测的局部解释。
   Use LIME for local explanations of single predictions.

## 目录 / TOC
1. [先建直觉：全局 vs 局部 ⭐](#1)
2. [💰 数据 + 训练黑箱模型](#2)
3. [置换重要性（全局）⭐](#3)
4. [PDP / ICE（全局趋势）⭐](#4)
5. [SHAP（全局+局部）⭐](#5)
6. [LIME（局部）+ 小结 ⭐](#6)


<a id="1"></a>
## 1. 先建直觉：全局 vs 局部 ⭐ / Intuition: Global vs Local

解释模型有两个层次（这是组织所有方法的关键框架）：
There are two levels of explanation (the key framework organizing all methods):
- **全局解释(global)**：整个模型**总体上**依赖哪些特征？哪些重要？某特征整体上怎么影响预测？——工具：置换重要性、PDP。
  **Global:** which features does the model rely on **overall**? Which matter? How does a feature affect predictions in general? — tools: permutation importance, PDP.
- **局部解释(local)**：对**这一条**具体预测，是哪些特征把它推高/推低的？——工具：LIME，以及 SHAP 的单样本解释。"为什么**这个**贷款被拒"就是局部问题。
  **Local:** for **this one** prediction, which features pushed it up/down? — tools: LIME, and SHAP's per-sample explanation. "Why was **this** loan rejected" is a local question.

**SHAP 的独特之处**：它**同时**给出局部（每条预测的特征贡献）和全局（把局部贡献聚合）解释，且有坚实的博弈论基础——这就是它成为行业标准的原因。
**SHAP's uniqueness:** it gives **both** local (per-prediction feature contributions) and global (aggregated) explanations, on a solid game-theory foundation — why it's the industry standard.


<a id="2"></a>
## 2. 数据 + 训练黑箱模型 / Data & a Black-box Model

用合成 **Adult Income**（同 5.8 风格）训一个梯度提升模型——它很准但不可解释。我们要解释它。
We train a gradient boosting model on synthetic **Adult Income** (5.8 style) — accurate but opaque. We'll explain it.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

def make_income(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n); edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years + 0.02*hours
             + 0.0002*np.sqrt(capital_gain)*edu_years*0.3 + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours, "capital_gain": capital_gain.round(0)})
    return X.astype(float), y      # 转 float: PDP 不支持整数列 / PDP needs float dtypes

X, y = make_income()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
model = GradientBoostingClassifier(n_estimators=150, max_depth=3, random_state=0).fit(X_tr, y_tr)
from sklearn.metrics import roc_auc_score
print(f"GBDT 高收入预测 test AUC = {roc_auc_score(y_te, model.predict_proba(X_te)[:,1]):.3f}")
print("模型很准, 但'为什么预测某人高收入'是黑箱 — 下面用解释工具打开它")


<a id="3"></a>
## 3. 置换重要性（全局）⭐ / Permutation Importance (Global)

**置换重要性**回答"哪个特征最重要"：把某一列的值**随机打乱**（破坏它和目标的关系），看模型性能掉多少——掉得越多说明模型越依赖它。它**模型无关、且在测试集上算**，比树自带的不纯度重要性（对高基数特征有偏，5.7）更可靠，是全局重要性的首选。
**Permutation importance** answers "which feature matters most": **shuffle one column** (destroying its link to the target) and see how much performance drops — a bigger drop means greater reliance. It's **model-agnostic and computed on test data**, more reliable than trees' built-in impurity importance (biased toward high-cardinality features, 5.7).


In [ ]:
from sklearn.inspection import permutation_importance

# 在 test 集上打乱每个特征, 重复 30 次取平均性能下降 / shuffle each feature, measure drop
perm = permutation_importance(model, X_te, y_te, n_repeats=30, random_state=0, scoring="roc_auc")
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.2))
imp.plot(kind="barh", xerr=perm.importances_std[np.argsort(perm.importances_mean)], ax=ax)
ax.invert_yaxis(); ax.set_xlabel("打乱后 AUC 下降量 (越大越重要)")
ax.set_title("置换重要性(全局): 打乱该特征, 模型掉多少分")
plt.tight_layout(); plt.show()
print("重要性排序:"); print(imp.round(4).to_string())
print("→ edu_years/age 最重要(打乱它们 AUC 掉最多); 模型无关 + test 集算 → 比不纯度重要性可靠")


<a id="4"></a>
## 4. PDP / ICE（全局趋势）⭐ / PDP / ICE

置换重要性只说"重不重要"，不说"**怎么**影响"。**部分依赖图(PDP)** 回答后者：固定其它特征，看某特征从小到大变化时，**平均预测**怎么变（是单调升？有拐点？）。**ICE（个体条件期望）** 是 PDP 的"未平均"版——每条线是一个样本，能揭示 PDP 的平均掩盖的**异质性/交互**。
Permutation importance says "how much it matters", not "**how** it affects". A **Partial Dependence Plot (PDP)** answers the latter: holding others fixed, how does the **average prediction** change as a feature varies (monotonic? a turning point?). **ICE (Individual Conditional Expectation)** is the un-averaged version — one line per sample, revealing **heterogeneity/interactions** that PDP's averaging hides.


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# PDP: age 和 edu_years 对预测的平均影响 / average effect of two features
PartialDependenceDisplay.from_estimator(model, X_te, ["age", "edu_years"], ax=axes)
axes[0].set_title("PDP: age (注意中年达峰的非线性)")
axes[1].set_title("PDP: edu_years (单调上升)")
plt.tight_layout(); plt.show()

# ICE: 每条线一个样本, 看 age 影响的个体差异 / per-sample curves
fig, ax = plt.subplots(figsize=(6.5, 4))
PartialDependenceDisplay.from_estimator(model, X_te.iloc[:200], ["age"], kind="both", ax=ax)  # both=ICE+PDP
ax.set_title("ICE(细线)+PDP(粗线): age\n个体曲线揭示平均之外的异质性")
plt.tight_layout(); plt.show()
print("PDP: edu_years 越高预测越高(单调); age 呈中年达峰(我们造数据时的-(age-45)²)")
print("ICE: 每条细线是一个样本, 形状不完全一致 → 存在与其它特征的交互(PDP 的平均会掩盖)")


<a id="5"></a>
## 5. SHAP（全局+局部）⭐ / SHAP (Global + Local)

**SHAP** 是当前最主流的解释工具，基于博弈论的 **Shapley 值**：把一次预测看成一场"合作博弈"，每个特征是一个"玩家"，**SHAP 值就是该特征对'这次预测偏离基线多少'的公平分摊**。它满足一组漂亮的公理（可加性等），且**局部贡献加总 = 全局重要性**——一套方法统一了两个层次。
**SHAP** is the leading interpretability tool, based on game-theory **Shapley values**: treat a prediction as a cooperative game where each feature is a "player", and **the SHAP value is that feature's fair share of "how far this prediction deviates from the baseline"**. It satisfies nice axioms (additivity, etc.), and **local contributions aggregate into global importance** — one method unifying both levels.

对树模型用 `TreeExplainer`（精确且快）。
For tree models use `TreeExplainer` (exact and fast).


In [ ]:
import shap

# 取一小批测试样本算 SHAP 值(树模型用 TreeExplainer, 精确快) / SHAP on a subset
X_sample = X_te.iloc[:300]
explainer = shap.TreeExplainer(model)
sv = explainer(X_sample)                       # 每个样本每个特征一个 SHAP 值 / per-sample values

# 全局: 平均 |SHAP值| = 全局重要性 / global importance = mean |SHAP|
global_imp = pd.Series(np.abs(sv.values).mean(0), index=X.columns).sort_values(ascending=False)
print("SHAP 全局重要性(平均|SHAP值|):"); print(global_imp.round(4).to_string())

fig = plt.figure()
shap.summary_plot(sv.values, X_sample, show=False)    # 蜂群图: 每点一样本, 颜色=特征值
plt.title("SHAP summary: 每点一样本, 横轴=对预测的推动, 颜色=特征值高低")
plt.tight_layout(); plt.show()
print("\n蜂群图读法: 点在右=把预测推向高收入; 红=特征值高")
print("如 edu_years 高(红点)普遍在右 → 高学历推高收入预测; 这是全局模式")


In [ ]:
# 局部: 解释单独一条预测, 看各特征怎么把它从基线推到最终值 / explain ONE prediction
i = 0
base = float(np.ravel(explainer.expected_value)[0])   # 基线可能是数组, 取标量 / coerce to scalar
print(f"解释第 {i} 个样本(特征: {X_sample.iloc[i].to_dict()}):")
print(f"  模型基线(平均 log-odds) = {base:.3f}")
contribs = pd.Series(sv.values[i], index=X.columns).sort_values(key=abs, ascending=False)
for f, v in contribs.items():
    print(f"  {f:<14} SHAP={v:+.3f}  ({'推高 ↑' if v>0 else '拉低 ↓'} 高收入概率)")
print(f"  基线 + 所有 SHAP 之和 = {base + sv.values[i].sum():.3f} = 该样本的预测(可加性!)")
print("→ 局部解释直接回答'为什么这个人被预测成(不)高收入', 各特征贡献可加")


<a id="6"></a>
## 6. LIME（局部）+ 小结 ⭐ / LIME (Local) & Summary

**LIME（局部可解释的模型无关解释）** 是另一种局部解释思路：想解释某条预测时，它在这条样本**附近随机扰动生成一堆点**，用黑箱模型给它们打分，然后**在这个局部邻域里拟合一个简单的线性模型**——用这个简单模型的系数来近似解释黑箱在这一点的行为。
**LIME (Local Interpretable Model-agnostic Explanations)** is another local approach: to explain a prediction, it **perturbs the sample to generate nearby points**, scores them with the black box, then **fits a simple linear model in that local neighborhood** — using the simple model's coefficients to approximate the black box's behavior at that point.

**SHAP vs LIME**（面试常问）：SHAP 有理论保证（Shapley 公理、可加、一致），全局局部统一，但较慢；LIME 更快更直观，但局部线性近似**不稳定**（换个扰动可能给不同解释）。**实战首选 SHAP**，LIME 作为快速补充。
**SHAP vs LIME** (often asked): SHAP has theoretical guarantees (Shapley axioms, additive, consistent), unifies global/local, but is slower; LIME is faster and intuitive but its local linear approximation is **unstable** (different perturbations can give different explanations). **SHAP is the practical default**; LIME a quick complement.


In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# LIME: 在样本邻域拟合局部线性模型来解释 / local linear surrogate
lime_exp = LimeTabularExplainer(X_tr.values, feature_names=list(X.columns),
                                class_names=["低收入","高收入"], mode="classification", random_state=0)
exp = lime_exp.explain_instance(X_te.iloc[0].values, model.predict_proba, num_features=4)
print(f"LIME 解释第 0 个样本(预测高收入概率={model.predict_proba(X_te.iloc[[0]])[0,1]:.2f}):")
for feat, weight in exp.as_list():
    print(f"  {feat:<24} 权重 weight={weight:+.3f}  ({'推高 ↑' if weight>0 else '拉低 ↓'})")
print("\nLIME 给出该点附近'简单线性近似'的特征权重 → 快但局部近似可能不稳")


```
全局解释(整个模型) vs 局部解释(单条预测) — 组织所有方法的框架
置换重要性(全局): 打乱某列看性能掉多少; 模型无关+test集算, 比不纯度重要性可靠(5.7)
PDP(全局趋势): 某特征如何影响平均预测; ICE: 每样本一条线, 揭示交互/异质性
SHAP(全局+局部统一): Shapley 值, 公平分摊每特征对'偏离基线'的贡献; 可加+有理论保证; 行业标准
LIME(局部): 邻域扰动+拟合局部线性模型; 快直观但不稳定
SHAP vs LIME: SHAP 有理论保证/统一/较慢, LIME 快/直观/不稳; 实战首选 SHAP
```

### 💡 面试速查 / Interview cheat-sheet
1. **全局(模型整体) vs 局部(单条预测)** 是组织解释方法的框架。
   Global (whole model) vs local (single prediction) organizes all methods.
2. **SHAP = Shapley 值**, 公平分摊特征贡献, 可加 + 全局局部统一, 行业标准。
   SHAP = Shapley values, fair feature attribution, additive, unifies global/local, industry standard.
3. **SHAP vs LIME**: SHAP 有理论保证/稳定/慢; LIME 快/直观/不稳。
   SHAP has guarantees/stable/slow; LIME is fast/intuitive/unstable.
4. **置换重要性比不纯度重要性可靠**(模型无关, test 集算)。
   Permutation importance beats impurity importance (model-agnostic, on test).
5. **PDP 看平均影响, ICE 看个体**(揭示交互)。
   PDP shows average effect; ICE shows individuals (reveals interactions).

### 下一节 / Next
**7.9 概率校准**——解释之后, 还要保证模型输出的概率"可信"。Platt/等张校准、可靠性曲线(系统化 5.15/4.16)。
**7.9 Calibration** — beyond explaining, ensure the model's probabilities are trustworthy. Platt/isotonic calibration and reliability diagrams (systematizing 5.15/4.16).
